<a href="https://colab.research.google.com/github/AITrading1995/AITrading1995/blob/main/Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git

In [83]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tvDatafeed import TvDatafeed, Interval
import plotly.graph_objects as go
import plotly.express as px
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

In [143]:
class Analysis:
    def __init__(self, symbol: list | str, exchange: str, time: str, bar: int) -> None:
        self.tv = TvDatafeed()
        self.symbols = symbol
        self.exchange = exchange
        self.time = time
        self.bar = bar

    def get_data(self):
        time_map = {
            '1m': Interval.in_1_minute,
            '5m': Interval.in_5_minute,
            '15m': Interval.in_15_minute,
            '30m': Interval.in_30_minute,
            '1h': Interval.in_1_hour,
            '4h': Interval.in_4_hour,
            'D': Interval.in_daily
        }

        try:
            interval = time_map.get(self.time)
            if interval is None:
                raise ValueError(f"Invalid time interval: {self.time}")

            if isinstance(self.symbols, list):
                data = {}
                for symbol in self.symbols:
                    df = self.tv.get_hist(
                        symbol=symbol,
                        exchange=self.exchange,
                        interval=interval,
                        n_bars=self.bar
                    )
                    if df is None or df.empty:
                        print(f"[WARNING] ไม่พบ symbol '{symbol}' ใน exchange '{self.exchange}'")
                    else:
                        data[symbol] = df
                if not data:
                    print("[ERROR] ไม่พบ symbol ใดเลยที่โหลดได้สำเร็จ")
                    return None
                return data
            else:
                df = self.tv.get_hist(
                    symbol=self.symbols,
                    exchange=self.exchange,
                    interval=interval,
                    n_bars=self.bar
                )
                if df is None or df.empty:
                    print(f"[ERROR] ไม่พบ symbol '{self.symbols}' ใน exchange '{self.exchange}'")
                    return None
                return {self.symbols: df}

        except Exception as e:
            print(f"[ERROR] {e}")
            return None

    def get_close(self) -> pd.DataFrame:
        data = self.get_data()
        if not data:
            return pd.DataFrame()
        return pd.DataFrame({sym: df['close'] for sym, df in data.items()})

    def log_return(self, cumsum=False) -> pd.DataFrame:
        data = self.get_data()
        if not data:
            return pd.DataFrame()
        log_ret = {}
        for sym, df in data.items():
            ret = np.log(df['close'] / df['close'].shift(1))
            if cumsum:
                ret = ret.cumsum()
            log_ret[sym] = ret
        return pd.DataFrame(log_ret)

    def volatility(self, plot: bool = False, param: int = 252):
      try:
          df = self.log_return()

          if df is None or df.empty:
              print("[ERROR] ไม่มีข้อมูล log return สำหรับคำนวณ volatility")
              return None

          # คำนวณ volatility (annualized)
          vol_df = df.dropna().rolling(window=param).std() * np.sqrt(param)

          if plot:
              fig = go.Figure()
              for col in vol_df.columns:
                  fig.add_trace(go.Scatter(
                      x=vol_df.index,
                      y=vol_df[col],
                      mode='lines',
                      name=col
                  ))

              fig.update_layout(
                  title=f"Annualized Volatility (Rolling {param} Days)",
                  xaxis_title="Date",
                  yaxis_title="Volatility",
                  template="plotly_dark",
                  height=500,
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
              )

              fig.show()
              return

          else:
              return vol_df

      except Exception as e:
          print(f"[ERROR] {e}")
          return None

    def log_plot(self, cummulate=False):
        df = self.log_return(cummulate)
        if df.empty:
            print("[ERROR] ไม่มีข้อมูล log return สำหรับ plot")
            return

        fig = go.Figure()
        for col in df.columns:
            fig.add_trace(go.Scatter(
                x=df.index,
                y=df[col],
                mode='lines',
                name=col
            ))

        fig.update_layout(
            title="Cumulative Log Return" if cummulate else "Log Return",
            xaxis_title="Datetime",
            yaxis_title="Log Return",
            template="plotly_dark",
            legend=dict(x=0, y=1)
        )
        fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])

        fig.show()


    # คำนวณ Correlation Matrix
    def correlation_matrix(self):
        df = self.get_close()
        return df.corr()

    # ทดสอบ Cointegration Matrix
    def cointegration_test(self):
        df = self.get_close().dropna() # Drop rows with NaN values
        symbols = df.columns
        result = pd.DataFrame(index=symbols, columns=symbols)

        for i in symbols:
            for j in symbols:
                if i == j:
                    result.loc[i, j] = np.nan
                else:
                    try:
                        _, pvalue, _ = coint(df[i], df[j])
                        result.loc[i, j] = pvalue
                    except ValueError as e:
                        print(f"[WARNING] ไม่สามารถคำนวณ cointegration test ระหว่าง {i} และ {j}: {e}")
                        result.loc[i, j] = np.nan

        return result.astype(float)


    # ตรวจสอบความมีนัยสำคัญ (p-value < 0.05 ถือว่ามีความสัมพันธ์แบบมีนัย)
    def significance_matrix(self, threshold=0.05):
        pval_matrix = self.cointegration_test()
        return (pval_matrix < threshold).astype(int)


    def plot_correlation_heatmap(self, method='pearson'):
        df = self.get_close()
        corr_matrix = df.corr(method=method)

        fig = px.imshow(
            corr_matrix,
            text_auto=True,
            color_continuous_scale='Viridis',
            title=f'{method.capitalize()} Correlation Heatmap',
        )
        fig.update_layout(width=800, height=700)
        fig.show()

    def summary(self):
        df = self.get_close()
        print('================ Summary ================')
        print('correlation_matrix')
        print(self.correlation_matrix())
        print('=========================================')
        print('cointegration_test')
        print(self.cointegration_test())
        print('=========================================')
        print('significance_matrix')
        print(self.significance_matrix())
        print('=========================================')

In [144]:
symbol , ex , time , bar = ['XAUUSD','GBPUSD','EURUSD','AUDNZD','USDJPY'],'OANDA','15m',6000
test = Analysis(symbol,ex,time,bar)
df = test.get_data()

In [145]:
close = test.get_close()

In [146]:
log_close = test.log_return()

In [147]:
test.log_plot(cummulate=True)

In [148]:
test.volatility(plot=True,param= 30)

In [149]:
test.plot_correlation_heatmap()


In [150]:
test.summary()

================ Summary ================
correlation_matrix
          XAUUSD    GBPUSD    EURUSD    AUDNZD    USDJPY
XAUUSD  1.000000 -0.088103 -0.040907  0.012495 -0.152102
GBPUSD -0.088103  1.000000  0.430436 -0.507146 -0.718587
EURUSD -0.040907  0.430436  1.000000  0.351065  0.141607
AUDNZD  0.012495 -0.507146  0.351065  1.000000  0.829309
USDJPY -0.152102 -0.718587  0.141607  0.829309  1.000000
cointegration_test
          XAUUSD    GBPUSD    EURUSD    AUDNZD    USDJPY
XAUUSD       NaN  0.016306  0.018165  0.020779  0.022366
GBPUSD  0.441225       NaN  0.699688  0.483136  0.310263
EURUSD  0.215280  0.336392       NaN  0.251415  0.217142
AUDNZD  0.978417  0.898305  0.950858       NaN  0.321519
USDJPY  0.577149  0.190060  0.471361  0.063715       NaN
significance_matrix
        XAUUSD  GBPUSD  EURUSD  AUDNZD  USDJPY
XAUUSD       0       1       1       1       1
GBPUSD       0       0       0       0       0
EURUSD       0       0       0       0       0
AUDNZD       0       0      

#Correlation Matrix
| คู่ที่มีความสัมพันธ์สูง (บวก) | ค่าความสัมพันธ์ |
| ----------------------------- | --------------- |
| AUDNZD - USDJPY               | `0.829`         |
| EURUSD - GBPUSD               | `0.430`         |
| EURUSD - AUDNZD               | `0.351`         |

| คู่ที่มีความสัมพันธ์ต่ำ (ลบ) | ค่าความสัมพันธ์ |
| ---------------------------- | --------------- |
| GBPUSD - USDJPY              | `-0.718`        |
| GBPUSD - AUDNZD              | `-0.507`        |

🔍 ตีความ:

AUDNZD กับ USDJPY มีพฤติกรรมราคาคล้ายกัน → เคลื่อนไหวในทิศทางเดียวกันบ่อย

GBPUSD กับ USDJPY มีความสัมพันธ์เชิงลบสูง → เป็นคู่ที่เคลื่อนไหวสวนทาง

#Cointegration Test (P-Value)
ค่าที่ใช้ตรวจสอบว่า ราคาปิดของสินทรัพย์ 2 ตัว มีความสัมพันธ์ในระยะยาวหรือไม่
(p-value ต่ำกว่า 0.05 แสดงว่า "มี cointegration")

| คู่ที่มี P-value ต่ำที่สุด (อาจมี cointegration) | P-Value |
| ------------------------------------------------ | ------- |
| XAUUSD - GBPUSD                                  | `0.016` |
| XAUUSD - EURUSD                                  | `0.018` |
| XAUUSD - AUDNZD                                  | `0.020` |
| XAUUSD - USDJPY                                  | `0.022` |

#Significance Matrix
แสดงว่า Cointegration มี นัยสำคัญทางสถิติหรือไม่ (1 = มีนัยสำคัญ, 0 = ไม่มี)

| คู่ที่สำคัญ           | ความหมาย      |
| --------------------- | ------------- |
| `XAUUSD - GBPUSD` → 1 | มีนัยสำคัญสูง |
| `XAUUSD - EURUSD` → 1 | มีนัยสำคัญสูง |
| `XAUUSD - AUDNZD` → 1 | มีนัยสำคัญสูง |
| `XAUUSD - USDJPY` → 1 | มีนัยสำคัญสูง |

เฉพาะ XAUUSD เท่านั้น ที่มีการ cointegrate กับคู่สกุลเงินอื่นๆ อย่างมีนัยสำคัญ

อาจแสดงว่า XAUUSD คือตัวนำทาง หรือมีบทบาทสำคัญในตลาดตอนนี้

#🎯 กลยุทธ์ที่สามารถนำไปใช้

| กลยุทธ์              | รายละเอียด                                                                |
| -------------------- | ------------------------------------------------------------------------- |
| **Pair Trading**     | ใช้กับคู่ที่มี cointegration เช่น XAUUSD - GBPUSD                         |
| **Mean Reversion**   | วิเคราะห์เมื่อ spread เบี่ยงจากค่าเฉลี่ย                                  |
| **Hedging Strategy** | ใช้คู่ที่มีความสัมพันธ์เชิงลบ เช่น GBPUSD - USDJPY เพื่อลดความเสี่ยงพอร์ต |


#✅ สรุปโดยรวม

XAUUSD มีความสัมพันธ์เชิงโครงสร้างกับหลายคู่ → น่าสนใจสำหรับกลยุทธ์ระยะยาว

มีคู่ที่เคลื่อนไหวสวนกัน (เช่น GBPUSD vs USDJPY) → ใช้ hedging ได้

EURUSD, AUDNZD, USDJPY มีบทบาทน้อยใน cointegration รอบนี้
